In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import os
from tqdm.auto import tqdm

# 1. Official Evaluation Metric
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    # Safety check to avoid division by zero
    if denom == 0.0: 
        return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# 2. Custom Callbacks
class LgbTqdmCallback:
    """A custom callback to show a progress bar for LightGBM boosting rounds."""
    def __init__(self, max_iter):
        self.max_iter = max_iter
        self.pbar = tqdm(total=max_iter, desc="Boosting Rounds", leave=False)
        
    def __call__(self, env):
        self.pbar.update(1)
        if env.evaluation_result_list:
            score = env.evaluation_result_list[0][2]
            self.pbar.set_postfix({'val_rmse': f'{score:.5f}'})
            
    def __del__(self):
        self.pbar.close()

# 3. Load Data
print("Loading data...")
base_path = "./" 
train = pd.read_parquet(os.path.join(base_path, "train.parquet"))
test = pd.read_parquet(os.path.join(base_path, "test.parquet"))

# 4. Preprocessing
cat_cols = ['code', 'sub_code', 'sub_category'] 
feature_cols = [f'feature_{alpha}' for alpha in 
                ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z'] + 
                ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj','ak','al','am','an','ao','ap','aq','ar','as','at','au','av','aw','ax','ay','az'] +
                ['ba','bb','bc','bd','be','bf','bg','bh','bi','bj','bk','bl','bm','bn','bo','bp','bq','br','bs','bt','bu','bv','bw','bx','by','bz'] +
                ['ca','cb','cc','cd','ce','cf','cg','ch']]

# Encode categorical strings
for col in cat_cols:
    le = LabelEncoder()
    full_data = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(full_data)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

test['y_target'] = 0.0

# Base LightGBM Parameters
n_estimators = 1000
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'verbosity': -1,
    'seed': 42,
    'n_jobs': -1
}

# 5. Train Per Horizon
horizons = train['horizon'].unique()
overall_val_scores = {}

for h in tqdm(horizons, desc="Overall Progress (Horizons)"):
    
    train_h = train[train['horizon'] == h].copy()
    test_h = test[test['horizon'] == h].copy()
    
    split_index = int(train_h['ts_index'].max() * 0.85)
    tr_df = train_h[train_h['ts_index'] <= split_index]
    val_df = train_h[train_h['ts_index'] > split_index]
    
    X_tr = tr_df[cat_cols + feature_cols + ['ts_index']]
    y_tr = tr_df['y_target']
    w_tr = tr_df['weight']
    
    X_val = val_df[cat_cols + feature_cols + ['ts_index']]
    y_val = val_df['y_target']
    w_val = val_df['weight']
    
    tqdm_cb = LgbTqdmCallback(max_iter=n_estimators)
    
    model = lgb.LGBMRegressor(**params, n_estimators=n_estimators)
    model.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_val, y_val)],
        eval_sample_weight=[w_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            tqdm_cb
        ]
    )
    
    tqdm_cb.pbar.close() 
    
    # Evaluate locally using your exact metric
    val_preds = model.predict(X_val)
    local_score = weighted_rmse_score(y_val.values, val_preds, w_val.values)
    overall_val_scores[h] = local_score
    
    tqdm.write(f"✓ Horizon {h} Validation Score: {local_score:.5f}")
    
    if len(test_h) > 0:
        X_test_h = test_h[cat_cols + feature_cols + ['ts_index']]
        test_preds = model.predict(X_test_h)
        test.loc[test['horizon'] == h, 'y_target'] = test_preds

# 6. Submission
print("\nOverall Local Validation Scores:", overall_val_scores)
print("Average Local Score:", np.mean(list(overall_val_scores.values())))

submission = test[['id', 'y_target']].copy()
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' saved!")

Loading data...


Overall Progress (Horizons):  25%|██▌       | 1/4 [00:07<00:21,  7.22s/it]

✓ Horizon 25 Validation Score: 0.17362


Overall Progress (Horizons):  50%|█████     | 2/4 [00:13<00:13,  6.74s/it]

✓ Horizon 1 Validation Score: 0.04241


Overall Progress (Horizons):  75%|███████▌  | 3/4 [00:20<00:07,  7.01s/it]

✓ Horizon 3 Validation Score: 0.07104


Overall Progress (Horizons): 100%|██████████| 4/4 [00:27<00:00,  6.77s/it]


✓ Horizon 10 Validation Score: 0.10444

Overall Local Validation Scores: {np.int32(25): 0.17361779299251281, np.int32(1): 0.04241389749406265, np.int32(3): 0.07103961391265835, np.int32(10): 0.10444439853760266}
Average Local Score: 0.09787892573420913
Submission file 'submission.csv' saved!
